In [0]:
# --- Schema & Messy Customer / Product / Transaction Data Generator ---
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random, string, datetime

# 1. Create Schema
spark.sql("CREATE SCHEMA IF NOT EXISTS visual_data_prep")

# ── Helper functions ──────────────────────────────────────────────────
def messy(val, miss_pct=0.20):
    """Randomly return None, or a messy version of the value."""
    if random.random() < miss_pct:
        return None
    return val

def messy_phone(country_code, miss_pct=0.25):
    codes = {
        "IN": lambda: random.choice([
            f"+91-{random.randint(7000000000,9999999999)}",
            f"091 {random.randint(7000000000,9999999999)}",
            f"{random.randint(7000000000,9999999999)}",
            f"+91 {random.randint(70000,99999)} {random.randint(10000,99999)}",
            "",  # blank
            "N/A",
        ]),
        "US": lambda: random.choice([
            f"+1-{random.randint(200,999)}-{random.randint(200,999)}-{random.randint(1000,9999)}",
            f"({random.randint(200,999)}) {random.randint(200,999)}-{random.randint(1000,9999)}",
            f"{random.randint(2002000000,9999999999)}",
            f"1{random.randint(2002000000,9999999999)}",
            "",
            "NA",
        ]),
        "UK": lambda: random.choice([
            f"+44 {random.randint(7000,7999)} {random.randint(100000,999999)}",
            f"0{random.randint(7000,7999)}{random.randint(100000,999999)}",
            f"44-{random.randint(7000,7999)}-{random.randint(100000,999999)}",
            "",
            "N.A.",
        ]),
    }
    if random.random() < miss_pct:
        return None
    return codes[country_code]()

def messy_email(first, last, miss_pct=0.25):
    if random.random() < miss_pct:
        return None
    domains = ["gmail.com", "yahoo.com", "outlook.com", "hotmail.com", "mail.com", "proton.me"]
    patterns = [
        f"{first.lower()}.{last.lower()}@{random.choice(domains)}",
        f"{first.lower()}{random.randint(1,999)}@{random.choice(domains)}",
        f"{first[0].lower()}{last.lower()}@{random.choice(domains)}",
        f"{first.lower()}_{last.lower()}@{random.choice(domains)}",
        f"{first.lower()}@{random.choice(domains)}",  # missing last name
        f"{first.lower()}.{last.lower()}@{random.choice(domains)}..",  # bad domain
        f"{first.lower()} {last.lower()}@{random.choice(domains)}",  # space in email
    ]
    return random.choice(patterns)

def messy_address_india():
    parts = [
        random.choice(["Flat", "Fl.", "FLT", "flat", ""]) + " " + str(random.randint(1,500)),
        random.choice([", ", "  ", " / ", "- "]),
        random.choice(["MG Road", "mg road", "M.G Road", "Mahatma Gandhi Rd", "Nehru Nagar", "Shivaji Marg", "Lal Bagh", "Station Rd.", "NH-44", "Sec-" + str(random.randint(1,60))]),
        random.choice([", ", "  ", " , ", ",,"]),
        random.choice(["Mumbai", "mumbai", "MUMBAI", "Delhi", "delhi", "Bangalore", "bangalore", "BENGALURU", "Hyderabad", "HYDERABAD", "Chennai", "Pune", "pune", "Kolkata", "Jaipur", "Lucknow", "Ahmedabad"]),
        random.choice([", ", " - ", "  ", ","]),
        random.choice(["Maharashtra", "MH", "mh", "Karnataka", "KA", "Telangana", "TG", "Tamil Nadu", "TN", "Delhi", "DL", "Rajasthan", "RJ", "UP", "WB", "GJ"]),
        " ",
        random.choice([str(random.randint(100000,999999)), "", "  ", "NA"]),
    ]
    if random.random() < 0.15:
        return None
    return "".join(parts)

def messy_address_usa():
    parts = [
        str(random.randint(1,9999)) + " ",
        random.choice(["Main St", "main street", "MAIN ST.", "Oak Ave", "oak avenue", "Elm Dr", "elm drive", "Maple Blvd", "Pine Lane", "5th Ave", "Broadway", "Washington Blvd"]),
        random.choice([", ", "  ", " , ", ",,", ", Apt " + str(random.randint(1,30)) + ", "]),
        random.choice(["New York", "new york", "NEW YORK", "Los Angeles", "Chicago", "chicago", "Houston", "Phoenix", "San Antonio", "Dallas", "San Jose", "Austin", "Jacksonville", "Seattle", "Denver", "Boston"]),
        random.choice([", ", " ", "  ", ","]),
        random.choice(["NY", "ny", "CA", "ca", "IL", "TX", "tx", "AZ", "FL", "WA", "CO", "MA", "GA"]),
        " ",
        random.choice([str(random.randint(10000,99999)), str(random.randint(10000,99999)) + "-" + str(random.randint(1000,9999)), "", "  ", "XXXXX"]),
    ]
    if random.random() < 0.15:
        return None
    return "".join(parts)

def messy_address_uk():
    parts = [
        str(random.randint(1,300)) + " ",
        random.choice(["High Street", "high st", "HIGH ST.", "Church Road", "Station Rd", "London Road", "Victoria Ave", "King's Road", "Queens Blvd", "Park Lane", "Baker Street"]),
        random.choice([", ", "  ", " , ", ",,"]),
        random.choice(["London", "london", "LONDON", "Manchester", "Birmingham", "BIRMINGHAM", "Leeds", "Glasgow", "Liverpool", "Edinburgh", "Bristol", "Sheffield", "Cardiff"]),
        random.choice([", ", " ", "  "]),
        random.choice([
            random.choice(["SW","NW","SE","EC","WC","E","W","N"]) + str(random.randint(1,20)) + " " + str(random.randint(1,9)) + random.choice(string.ascii_uppercase) + random.choice(string.ascii_uppercase),
            "",
            "N/A",
            random.choice(["M","B","LS","G","L","EH","BS","S","CF"]) + str(random.randint(1,15)) + " " + str(random.randint(1,9)) + random.choice(string.ascii_uppercase) + random.choice(string.ascii_uppercase),
        ]),
    ]
    if random.random() < 0.15:
        return None
    return "".join(parts)

# ── 2. CUSTOMER_IND (India) ──────────────────────────────────────────
ind_first = ["Aarav","Vivaan","Aditya","Vihaan","Arjun","Sai","Reyansh","Ayaan","Krishna","Ishaan",
             "Priya","Ananya","Diya","Aadhya","Meera","Riya","Saanvi","Kavya","Pooja","Neha"]
ind_last = ["Sharma","Patel","Reddy","Kumar","Singh","Nair","Gupta","Verma","Joshi","Iyer",
            "Das","Mehta","Rao","Pillai","Bhat","Mishra","Kapoor","Malik","Chopra","Thakur"]

customer_ind_data = []
for i in range(1, 21):
    fn = ind_first[i-1]
    ln = random.choice(ind_last)
    age = random.randint(18, 72)
    gender = random.choice(["Male", "Female", "M", "F", "male", "female", None, "Other"])
    customer_ind_data.append((
        f"IND-{1000+i}",
        fn if random.random() > 0.05 else fn.upper(),
        ln if random.random() > 0.05 else ln.lower(),
        messy_email(fn, ln),
        messy_phone("IN"),
        messy_address_india(),
        messy("India", 0.1),
        messy(str(age), 0.1),
        gender,
        messy(random.choice(["Active","active","ACTIVE","Inactive","inactive","Suspended","SUSPENDED"]), 0.1),
        messy(random.choice(["Gold","Silver","Bronze","Platinum","gold","SILVER"]), 0.15),
        messy(datetime.date(random.randint(2018,2024), random.randint(1,12), random.randint(1,28)).isoformat(), 0.05),
        messy(random.choice(["Online","In-Store","Referral","Social Media","online","referral",""]), 0.1),
    ))

schema_cust = StructType([
    StructField("customer_id", StringType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("email", StringType()),
    StructField("phone", StringType()),
    StructField("address", StringType()),
    StructField("country", StringType()),
    StructField("age", StringType()),
    StructField("gender", StringType()),
    StructField("status", StringType()),
    StructField("loyalty_tier", StringType()),
    StructField("registration_date", StringType()),
    StructField("acquisition_channel", StringType()),
])

spark.createDataFrame(customer_ind_data, schema_cust) \
    .write.mode("overwrite").saveAsTable("visual_data_prep.customer_ind")
print("✅ customer_ind created (20 rows)")

# ── 3. CUSTOMER_USA ──────────────────────────────────────────────────
us_first = ["James","Mary","Robert","Patricia","John","Jennifer","Michael","Linda","David","Elizabeth",
            "William","Barbara","Richard","Susan","Joseph","Jessica","Thomas","Sarah","Charles","Karen"]
us_last = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis","Rodriguez","Martinez",
           "Hernandez","Lopez","Wilson","Anderson","Thomas","Taylor","Moore","Jackson","Martin","Lee"]

customer_usa_data = []
for i in range(1, 21):
    fn = us_first[i-1]
    ln = random.choice(us_last)
    age = random.randint(18, 75)
    gender = random.choice(["Male","Female","M","F","male",None,"Non-binary","Prefer not to say"])
    customer_usa_data.append((
        f"USA-{2000+i}",
        fn if random.random() > 0.05 else fn.lower(),
        ln if random.random() > 0.05 else ln.upper(),
        messy_email(fn, ln),
        messy_phone("US"),
        messy_address_usa(),
        messy(random.choice(["USA","US","United States","usa","U.S.A.","united states"]), 0.05),
        messy(str(age), 0.1),
        gender,
        messy(random.choice(["Active","active","ACTIVE","Inactive","Closed","closed"]), 0.1),
        messy(random.choice(["Gold","Silver","Bronze","Platinum","Diamond","GOLD"]), 0.15),
        messy(datetime.date(random.randint(2018,2024), random.randint(1,12), random.randint(1,28)).isoformat(), 0.05),
        messy(random.choice(["Online","In-Store","Referral","Social Media","Email Campaign","online"]), 0.1),
    ))

spark.createDataFrame(customer_usa_data, schema_cust) \
    .write.mode("overwrite").saveAsTable("visual_data_prep.customer_usa")
print("✅ customer_usa created (20 rows)")

# ── 4. CUSTOMER_UK ───────────────────────────────────────────────────
uk_first = ["Oliver","Amelia","George","Isla","Harry","Ava","Noah","Emily","Jack","Sophia",
            "Leo","Grace","Oscar","Mia","Charlie","Poppy","Jacob","Ella","Henry","Lily"]
uk_last = ["Smith","Jones","Taylor","Brown","Williams","Wilson","Johnson","Davies","Patel","Robinson",
           "Wright","Thompson","Evans","Walker","White","Roberts","Green","Hall","Wood","Harris"]

customer_uk_data = []
for i in range(1, 21):
    fn = uk_first[i-1]
    ln = random.choice(uk_last)
    age = random.randint(18, 78)
    gender = random.choice(["Male","Female","M","F",None,"Other","male"])
    customer_uk_data.append((
        f"UK-{3000+i}",
        fn if random.random() > 0.05 else fn.upper(),
        ln if random.random() > 0.05 else ln.lower(),
        messy_email(fn, ln),
        messy_phone("UK"),
        messy_address_uk(),
        messy(random.choice(["UK","United Kingdom","uk","U.K.","Great Britain","GB","England"]), 0.05),
        messy(str(age), 0.1),
        gender,
        messy(random.choice(["Active","active","Inactive","INACTIVE","Dormant","dormant"]), 0.1),
        messy(random.choice(["Gold","Silver","Bronze","Platinum","silver","BRONZE"]), 0.15),
        messy(datetime.date(random.randint(2018,2024), random.randint(1,12), random.randint(1,28)).isoformat(), 0.05),
        messy(random.choice(["Online","In-Store","Referral","Social Media","Partner","online"]), 0.1),
    ))

spark.createDataFrame(customer_uk_data, schema_cust) \
    .write.mode("overwrite").saveAsTable("visual_data_prep.customer_uk")
print("✅ customer_uk created (20 rows)")

# ── 5. PRODUCTS TABLE (30 products, messy descriptions) ─────────────
categories = {
    "Electronics": [
        ("Wireless Bluetooth Headphones", 49.99, "Over-ear headphons with noice cancellation, 30hr batt life. Compatable with all devces. Colr: Black/Wht"),
        ("Smart Watch Pro X", 199.99, "Fitness trackr with heartrate moniter, GPS, waterprof IP68. Works w/ iOS & andriod. Screen: 1.4inch AMLED"),
        ("USB-C Fast Charger 65W", 29.99, "GaN charger - supports PD3.0, QC4+. For laptps, phones, tablts. Input: 100-240V~50/60Hz  Weight: 120gm approx."),
        ("Portable Bluetooth Speaker", 39.99, "360° surond sound, IPX7 watrproof. Battry: 12hrs. Bluetooth 5.3. Dimensons: 7.5 x 7.5 x 8.2 cm. Wt: 540g"),
        ("Laptop Stand Adjustable", 34.99, "Ergonomc aluminium stand, fits 10-17\" laptps. Hieght adjustible 6 levels. Anti-slip sillicone pads."),
        ("Wireless Mouse Ergonomic", 24.99, "2.4GHz wireless + BT 5.0 dual mode. DPI: 800/1200/1600/2400. Battry: 2x AAA (not inclded)"),
    ],
    "Clothing": [
        ("Men's Cotton T-Shirt", 19.99, "100% orgnic cotton, pre-shrunk. Sizes: S/M/L/XL/XXL. Machine washble. Colors avialable: Blk, Wht, Navy, Grey"),
        ("Women's Running Shoes", 79.99, "Lightweight mesh uper, EVA midsole. Sizes: 5-11 US. Breatheable design. Suitble for road & trail. Wt: 230g per shoe"),
        ("Unisex Hoodie Fleece", 44.99, "80% cotton 20% polyster blend. Kangaroo pockt, drawstring hood. Oversized fit. Machine wash cold, tumble dry lo"),
        ("Kids Winter Jacket", 54.99, "Waterprof outer shell, fleece lining. Ages: 4-12yrs. Reflective strps for saftey. Hood detachble. Temp rating: -10°C"),
        ("Silk Scarf Premium", 29.99, "100% mulbery silk, hand-rolled edges. Dimensions: 90x90cm. Dry clean onlt. Multiple paterns available"),
    ],
    "Home & Kitchen": [
        ("Stainless Steel Cookware Set 10pc", 129.99, "18/10 stnless steel, tri-ply bottom. Includes: 2 frypans, 2 saucpans w/ lids, 1 stockpot w/ lid, 1 steamer. Dishwahser safe. Induction compatable."),
        ("Air Purifier HEPA H13", 149.99, "Covers upto 500 sq ft. 3-stage filtraton: pre-filter + HEPA H13 + activated carbon. Noise: 24-50dB. Auto mode w/ air quality sensr"),
        ("Electric Kettle 1.7L", 34.99, "Stainles steel interor, BPA-free. Auto shut-off, boil-dry protction. 1500W rapid boil. 360° swivl base"),
        ("Memory Foam Pillow", 39.99, "CertiPUR-US certfied foam. Hypoallergnic, dust mite resistent. Dimensions: 24x16x5.5in. Removble washable covr"),
        ("Robot Vacuum Cleaner", 249.99, "2500Pa suction, LiDAR navigaton. 150min runtime. App control (iOS/Andoid). Auto-empty staion compatible. HEPA filtr"),
    ],
    "Books": [
        ("Data Engineering Fundamentals", 42.99, "Hardcovr, 480 pgs. ISBN: 978-1234567890. Author: J. Smith. Topics: ETL, data piplines, warehousing, lake architecture. Publshed: 2023"),
        ("Machine Learning in Practice", 49.99, "Paperbck, 560 pgs. Covers supervized & unsupervised lerning, deep learning, NLP. With Python code exmples. 2nd editon"),
        ("Python Cookbook Advanced", 39.99, "3rd Ed., 650+ recipies. From basics to metaprograming. O'Reily publicaton. Kindle + paperbck available"),
    ],
    "Health & Beauty": [
        ("Organic Face Moisturizer", 24.99, "50ml. Ingrediants: Aloe vera, hyaluronic acid, vitamin E, jojba oil. Paraben-free, crueltry-free. For all skin typs. Shelf life: 24 mnths"),
        ("Electric Toothbrush Sonic", 59.99, "5 brushng modes, 2min smart timer. IPX7 waterpoof. Includes: 3 brush heads, travel case, USB chargng base. Battry: 30 days"),
        ("Vitamin D3 Supplements", 14.99, "120 softgels, 2000 IU per servng. Non-GMO, gluten free. 3rd party tested. Expiry: see bottl. Store below 25°C"),
    ],
    "Sports & Outdoors": [
        ("Yoga Mat Premium 6mm", 29.99, "Non-slip TPE material, eco-freindly. Dimensions: 183x61x0.6cm. Weight: 1.2kg. Includes carry strp. Latex-free"),
        ("Camping Tent 4-Person", 159.99, "Waterprof rating: 3000mm. Setup time: ~10min. Weight: 4.5kg packed. Includes rainfly, stakes, carry bag. Seams: factory sealed"),
        ("Resistance Bands Set", 19.99, "5 bands (10-50lbs), 2 handles, door anchor, ankle straps, carry bag. Latex. Sutable for home gym & physio therapy"),
        ("Insulated Water Bottle 1L", 22.99, "Dubble-wall vaccum insulation. Keeps cold 24hr / hot 12hr. BPA-free. Leak-proof lid. Material: 18/8 stainles steel"),
    ],
}

product_data = []
pid = 5000
for cat, items in categories.items():
    for name, price, desc in items:
        pid += 1
        # Add extra messiness to description
        messy_desc = desc
        if random.random() < 0.3:
            messy_desc = messy_desc + "  " + random.choice(["...", "(details TBC)", "[DRAFT]", "\n\n", "  "])
        if random.random() < 0.2:
            messy_desc = "  " + messy_desc  # leading whitespace
        
        product_data.append((
            f"PRD-{pid}",
            name if random.random() > 0.1 else name.upper(),
            cat if random.random() > 0.1 else cat.lower(),
            random.choice(["Electronics Co.","FashionHub","HomeGoods Inc.","BookWorld","HealthPlus","SportMax",
                           "electronics co.","FASHIONHUB",None,"Unknown","TBD"]),
            round(price * random.uniform(0.8, 1.2), 2),
            messy(round(price * random.uniform(1.5, 3.0), 2), 0.15),  # MRP/list price (sometimes missing)
            messy_desc,
            messy(str(random.randint(0, 500)), 0.1),  # stock_quantity
            messy(round(random.uniform(1.0, 5.0), 1), 0.1),  # rating
            messy(random.randint(0, 2000), 0.15),  # review_count
            messy(random.choice(["In Stock","in stock","Out of Stock","out of stock","Limited","Discontinued","PRE-ORDER"]), 0.1),
            random.choice(["kg","g","lbs","oz",None,""]),
            messy(round(random.uniform(0.1, 10.0), 2), 0.2),  # weight_value
            messy(random.choice(["New","Refurbished","Used - Like New","new","NEW",None]), 0.1),
        ))

schema_prod = StructType([
    StructField("product_id", StringType()),
    StructField("product_name", StringType()),
    StructField("category", StringType()),
    StructField("brand", StringType()),
    StructField("unit_price", DoubleType()),
    StructField("list_price", DoubleType()),
    StructField("description", StringType()),
    StructField("stock_quantity", StringType()),
    StructField("rating", DoubleType()),
    StructField("review_count", IntegerType()),
    StructField("availability_status", StringType()),
    StructField("weight_unit", StringType()),
    StructField("weight_value", DoubleType()),
    StructField("condition", StringType()),
])

spark.createDataFrame(product_data, schema_prod) \
    .write.mode("overwrite").saveAsTable("visual_data_prep.products")
print(f"✅ products created ({len(product_data)} rows)")

# ── 6. TRANSACTIONS TABLE (120+ transactions with timestamps) ───────
all_customer_ids = [f"IND-{1000+i}" for i in range(1,21)] + \
                   [f"USA-{2000+i}" for i in range(1,21)] + \
                   [f"UK-{3000+i}"  for i in range(1,21)]
all_product_ids  = [f"PRD-{5001+i}" for i in range(len(product_data))]

payment_methods = ["Credit Card","credit card","Debit Card","PayPal","paypal","UPI","Bank Transfer",
                   "Cash on Delivery","COD","Apple Pay","Google Pay","gpay",None,""]
order_statuses  = ["Completed","completed","COMPLETED","Pending","pending","Shipped","shipped",
                   "Delivered","Cancelled","cancelled","Refunded","REFUNDED","Processing","Failed"]
currencies      = ["INR","USD","GBP","inr","usd","gbp","$","£","₹",None]
channels        = ["Web","web","Mobile App","mobile app","In-Store","in-store","Phone","Marketplace",None]
shipping        = ["Standard","Express","Next Day","Free Shipping","Pickup","standard","express",None,"N/A"]
coupons         = ["SAVE10","SUMMER20","FLAT15","WELCOME5","NEWYEAR25","FREE_SHIP",None,None,None,None,None]  # mostly None

transaction_data = []
for i in range(1, 131):  # 130 transactions
    cust_id = random.choice(all_customer_ids)
    prod_id = random.choice(all_product_ids)
    qty = random.randint(1, 5)
    unit_price = round(random.uniform(10.0, 300.0), 2)
    discount = round(random.uniform(0, 0.30), 2) if random.random() > 0.4 else 0.0
    subtotal = round(unit_price * qty, 2)
    discount_amt = round(subtotal * discount, 2)
    tax = round((subtotal - discount_amt) * random.uniform(0.05, 0.20), 2)
    total = round(subtotal - discount_amt + tax, 2)
    
    # Random timestamp between 2022-01-01 and 2024-12-31
    start_ts = datetime.datetime(2022, 1, 1)
    end_ts   = datetime.datetime(2024, 12, 31, 23, 59, 59)
    delta    = end_ts - start_ts
    rand_ts  = start_ts + datetime.timedelta(seconds=random.randint(0, int(delta.total_seconds())))
    
    currency = "INR" if "IND" in cust_id else ("USD" if "USA" in cust_id else "GBP")
    if random.random() < 0.15:
        currency = random.choice(currencies)  # inject messy currency
    
    transaction_data.append((
        f"TXN-{10000+i}",
        cust_id,
        prod_id,
        rand_ts,  # order_timestamp
        qty,
        unit_price,
        subtotal,
        round(discount * 100, 1),  # discount_pct
        discount_amt,
        tax,
        total,
        random.choice(payment_methods),
        random.choice(order_statuses),
        currency,
        random.choice(channels),
        random.choice(shipping),
        random.choice(coupons),
        messy(random.choice(["Standard","Extended","No Warranty","1-Year","2-Year",None]), 0.3),
        messy(round(random.uniform(1.0, 5.0), 1), 0.25),  # customer_rating of transaction
        messy(rand_ts + datetime.timedelta(days=random.randint(1, 14)), 0.2),  # delivery_timestamp
    ))

schema_txn = StructType([
    StructField("transaction_id", StringType()),
    StructField("customer_id", StringType()),
    StructField("product_id", StringType()),
    StructField("order_timestamp", TimestampType()),
    StructField("quantity", IntegerType()),
    StructField("unit_price", DoubleType()),
    StructField("subtotal", DoubleType()),
    StructField("discount_pct", DoubleType()),
    StructField("discount_amount", DoubleType()),
    StructField("tax_amount", DoubleType()),
    StructField("total_amount", DoubleType()),
    StructField("payment_method", StringType()),
    StructField("order_status", StringType()),
    StructField("currency", StringType()),
    StructField("channel", StringType()),
    StructField("shipping_method", StringType()),
    StructField("coupon_code", StringType()),
    StructField("warranty", StringType()),
    StructField("customer_rating", DoubleType()),
    StructField("delivery_timestamp", TimestampType()),
])

spark.createDataFrame(transaction_data, schema_txn) \
    .write.mode("overwrite").saveAsTable("visual_data_prep.transactions")
print(f"✅ transactions created ({len(transaction_data)} rows)")

# ── Summary ──────────────────────────────────────────────────────────
print("\n" + "="*60)
print("📊 VISUAL DATA PREP SCHEMA — ALL TABLES READY")
print("="*60)
for t in ["customer_ind","customer_usa","customer_uk","products","transactions"]:
    cnt = spark.table(f"visual_data_prep.{t}").count()
    print(f"  visual_data_prep.{t:20s} → {cnt} rows")
print("="*60)
print("\n🔗 Relationships:")
print("  customers (customer_id) ←→ transactions (customer_id)")
print("  products  (product_id)  ←→ transactions (product_id)")
print("\n🧹 Messy data includes:")
print("  • Missing emails, phones, addresses (NULLs)")
print("  • Inconsistent casing (Mumbai/mumbai/MUMBAI)")
print("  • Inconsistent formats (phone: +91-xxx vs 091 xxx)")
print("  • Typos in product descriptions")
print("  • Mixed country representations (UK/United Kingdom/GB)")
print("  • Dirty currency codes ($, £, ₹ mixed with ISO codes)")
print("  • Inconsistent status values (Active/active/ACTIVE)")
print("  • Leading whitespace & trailing junk in descriptions")

In [0]:
tables_to_drop = [
    "visual_data_prep.bad_records_customer_ind",
    "visual_data_prep.bad_records_customer_uk",
    "visual_data_prep.bad_records_customer_usa",
    "visual_data_prep.bad_records_products",
    "visual_data_prep.bad_records_transactions",
    "visual_data_prep.customer_ind_clean",
    "visual_data_prep.customer_uk_clean",
    "visual_data_prep.customer_usa_clean",
    "visual_data_prep.products_clean",
    "visual_data_prep.transactions_clean",
]

for table in tables_to_drop:
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    print(f"✅ Dropped {table}")